# Run Three-Version Benchmark

**Purpose:**
- Use the T112A shared runner to run selected dataset rows through baseline, dflash, and cc-dflash.
- Safely manage configuration so execution doesn't unexpectedly load models without permission.

In [ ]:
import sys
import os
import json
from pathlib import Path
from datetime import datetime, timezone
import pandas as pd

_cwd = Path.cwd()
if _cwd.name != "notebooks":
    sys.path.insert(0, str(_cwd / "notebooks"))

import notebook_setup
ROOT = notebook_setup.setup_root()


## Configuration

In [ ]:
DATASET = "gsm8k"
LIMIT = 1
SEED = 42
MAX_NEW_TOKENS = 128
CONDITIONS = [
    "baseline_ar",
    "dflash_r1",
    "cc_dflash_r2",
]

## Execute Models

In [ ]:
from ccdf.demo import DemoRunner
from ccdf.config import load_config
from ccdf.demo.contracts import RunRequest
from ccdf.demo.writers import write_jsonl_append, write_flat_csv_append, write_json
from ccdf.demo.adapters.gsm8k import gsm8k_row_to_request
from ccdf.demo.adapters.qmsum import qmsum_row_to_request

try:
    config = load_config("config.yml")
except FileNotFoundError:
    config = {}

if isinstance(config, dict):
    config["dry_run"] = os.environ.get("CCDF_NOTEBOOK_TEST_MODE") == "1"
else:
    config.dry_run = os.environ.get("CCDF_NOTEBOOK_TEST_MODE") == "1"

runner = DemoRunner(config)

filename = "gsm8k_100.jsonl" if DATASET == "gsm8k" else "qmsum_meeting_qa_100.jsonl"
eval_path = ROOT / "data/eval" / filename
if not eval_path.exists():
    raise FileNotFoundError(f"Missing data: {eval_path}")

with open(eval_path, "r", encoding="utf-8") as f:
    rows = [json.loads(line) for line in f if line.strip()]
rows = rows[:LIMIT]

RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
out_dir = ROOT / "results/charts/notebook_demo"
runs_dir = out_dir / "runs" / RUN_ID
tables_dir = out_dir / "tables" / RUN_ID
summaries_dir = out_dir / "summaries" / RUN_ID

runs_dir.mkdir(parents=True, exist_ok=True)
tables_dir.mkdir(parents=True, exist_ok=True)
summaries_dir.mkdir(parents=True, exist_ok=True)

out_jsonl = runs_dir / f"{DATASET}_three_version.jsonl"
out_csv = tables_dir / f"{DATASET}_three_version.csv"

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", None)
pd.set_option("display.max_colwidth", None)
pd.set_option("display.width", None)

print(f"Run ID: {RUN_ID}")
print(f"Dataset: {DATASET}")
print(f"Rows: {len(rows)}\n")

for i, row in enumerate(rows):
    for cond_idx, cond in enumerate(CONDITIONS):
        print(f"[{cond_idx+1}/{len(CONDITIONS)}] Running {cond}...")
        try:
            if DATASET == "gsm8k":
                req = gsm8k_row_to_request(row, cond, seed=SEED, max_new_tokens=MAX_NEW_TOKENS)
            else:
                req = qmsum_row_to_request(row, cond, seed=SEED, max_new_tokens=MAX_NEW_TOKENS)
            
            res = runner.run(req)
            
            write_jsonl_append(res, out_jsonl)
            write_flat_csv_append(res, out_csv)
            
            print(f"[{cond_idx+1}/{len(CONDITIONS)}] {cond} complete.\n")
            
            print("--- Output Details ---")
            print(f"Fixture: {req.fixture_id}")
            print(f"Profile: {req.prompt_profile}")
            print(f"Condition: {cond}")
            print("\nPrompt:")
            print(req.prompt)
            if req.reference_answer:
                print(f"\nReference Answer: {req.reference_answer}")
            print("\nGenerated:")
            print(res.response.generated_text)
            print(f"\nFinish Reason: {res.response.finish_reason}")
            print(f"Output Tokens: {res.response.output_tokens}")
            print(f"E2E Latency (ms): {res.timing_ms.e2e:.2f}")
            print(f"Generation Throughput (tok/s): {res.throughput.generation_tok_s:.2f}")
            print(f"Peak VRAM (GiB): {res.resources.peak_vram_gib:.2f}")
            if res.quality.evaluation_status != "not_evaluated":
                print(f"Quality (Match): {res.quality.numeric_match}")
            print("----------------------\n")
            
        except Exception as e:
            print(f"[{cond_idx+1}/{len(CONDITIONS)}] Failed {cond}: {e}\n")

summary = {
    "dataset": DATASET,
    "limit": LIMIT,
    "conditions_run": CONDITIONS,
    "jsonl_path": str(out_jsonl.relative_to(ROOT)),
    "csv_path": str(out_csv.relative_to(ROOT))
}
write_json(summary, summaries_dir / f"{DATASET}_three_version_summary.json", overwrite=True)
write_json({
  "run_id": RUN_ID,
  "dataset": DATASET,
  "run_dir": str(runs_dir.relative_to(ROOT)),
  "table_dir": str(tables_dir.relative_to(ROOT)),
  "summary_dir": str(summaries_dir.relative_to(ROOT)),
  "figure_dir": f"results/charts/notebook_demo/figures/{RUN_ID}",
  "completed": True
}, out_dir / "latest_run.json", overwrite=True)

print("--- Final Comparison Table ---")
df = pd.read_csv(out_csv)
cols = ["condition", "condition_display_name", "fixture_id", "prompt_profile", 
        "original_input_tokens", "compressed_input_tokens", "compression_ratio", 
        "output_tokens", "t_compress_ms", "t_prefill_ms", "t_generation_ms", 
        "t_e2e_ms", "generation_tok_s", "e2e_tok_s", "peak_vram_gib", 
        "finish_reason", "evaluation_status", "numeric_match", "ok", "error_type"]
# Only include columns that actually exist in the CSV
disp_cols = [c for c in cols if c in df.columns]
display(df[disp_cols])

print("\nArtifacts saved:")
print(f"- {out_jsonl}")
print(f"- {out_csv}")
print(f"- {summaries_dir / f'{DATASET}_three_version_summary.json'}")
print(f"- {out_dir / 'latest_run.json'}")


## Interpretation Notes
- For GSM8K, the output may show a numeric exact match as a proxy for correctness.
- For QMSum, **Semantic correctness is not claimed; quality fields are bounded proxies only.**